<a href="https://colab.research.google.com/github/Papaloy1/2.2-actividad/blob/main/Sesi%C3%B3n_3_DFS_E_IDDFS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ACTIVIDAD A3 · SESIÓN 3 · DFS E IDDFS

# Grafo común de clase. Cada par contiene (sucesor, costo del paso).
GRAFO_EJEMPLO = {
    "S": [("A", 2), ("B", 1)],
    "A": [("G", 9), ("C", 2)],
    "B": [("D", 2)],
    "C": [("G", 2)],
    "D": [("S", 1), ("G", 5)],  # D → S forma un ciclo
    "G": []
}


def crear_resultado(ruta, costo, expandidos, frontera_maxima):
    """Reúne los datos que compararemos entre algoritmos."""
    return {
        "ruta": ruta,
        "profundidad": len(ruta) - 1,
        "costo": costo,
        "expandidos": expandidos,
        "frontera_maxima": frontera_maxima
    }


def dfs(grafo, inicio, meta):
    """Busca en profundidad usando una pila."""
    # Cada elemento guarda: estado actual, ruta seguida y costo acumulado.
    frontera = [(inicio, [inicio], 0)]
    explorados = set()
    expandidos = 0
    frontera_maxima = 1

    while frontera:  # Continúa mientras queden nodos pendientes.
        estado, ruta, costo = frontera.pop()  # LIFO: sale el último agregado.

        if estado in explorados:
            continue  # Evita procesar otra vez un estado.

        if estado == meta:
            return crear_resultado(
                ruta, costo, expandidos, frontera_maxima
            )

        explorados.add(estado)
        expandidos += 1

        # Se apilan en el orden escrito. Por LIFO, saldrá primero
        # el último sucesor de la lista.
        for vecino, costo_paso in grafo.get(estado, []):
            if vecino not in explorados:
                frontera.append((
                    vecino,
                    ruta + [vecino],
                    costo + costo_paso
                ))

        frontera_maxima = max(frontera_maxima, len(frontera))

    return None  # Se agotó la frontera sin encontrar la meta.


def busqueda_limitada(grafo, inicio, meta, limite):
    """Hace una búsqueda en profundidad sin superar el límite indicado."""
    frontera = [(inicio, [inicio], 0)]
    expandidos = 0
    frontera_maxima = 1

    while frontera:
        estado, ruta, costo = frontera.pop()
        profundidad = len(ruta) - 1

        if estado == meta:
            resultado = crear_resultado(
                ruta, costo, expandidos, frontera_maxima
            )
            return resultado, expandidos, frontera_maxima

        if profundidad == limite:
            continue  # Aquí se detiene esta rama; no genera sucesores.

        expandidos += 1

        # reversed permite visitar primero el sucesor escrito primero.
        for vecino, costo_paso in reversed(grafo.get(estado, [])):
            if vecino not in ruta:  # Evita ciclos dentro de esta ruta.
                frontera.append((
                    vecino,
                    ruta + [vecino],
                    costo + costo_paso
                ))

        frontera_maxima = max(frontera_maxima, len(frontera))

    return None, expandidos, frontera_maxima


def iddfs(grafo, inicio, meta):
    """Repite la búsqueda limitada con límites 0, 1, 2..."""
    estados = set(grafo) | {inicio}
    for sucesores in grafo.values():
        estados.update(vecino for vecino, _ in sucesores)

    expandidos_total = 0
    frontera_maxima_total = 1

    # En un grafo finito, una ruta simple usa como máximo
    # número_de_estados - 1 transiciones.
    for limite in range(len(estados)):
        resultado, expandidos, frontera_maxima = busqueda_limitada(
            grafo, inicio, meta, limite
        )

        expandidos_total += expandidos
        frontera_maxima_total = max(
            frontera_maxima_total, frontera_maxima
        )

        if resultado is not None:
            resultado["expandidos"] = expandidos_total
            resultado["frontera_maxima"] = frontera_maxima_total
            resultado["limite_encontrado"] = limite
            return resultado

    return None


def mostrar(nombre, resultado):
    """Imprime una salida legible."""
    print(f"\n{nombre}")
    if resultado is None:
        print("No se encontró una ruta.")
        return

    print("Ruta:", " → ".join(resultado["ruta"]))
    print("Profundidad:", resultado["profundidad"])
    print("Costo:", resultado["costo"])
    print("Nodos expandidos:", resultado["expandidos"])
    print("Tamaño máximo de la frontera:", resultado["frontera_maxima"])

    if "limite_encontrado" in resultado:
        print("Límite en que se encontró la meta:",
              resultado["limite_encontrado"])


# COMPROBACIÓN AUTOMÁTICA CON EL GRAFO COMÚN
resultado_dfs = dfs(GRAFO_EJEMPLO, "S", "G")
resultado_iddfs = iddfs(GRAFO_EJEMPLO, "S", "G")

assert resultado_dfs["ruta"] == ["S", "B", "D", "G"]
assert resultado_dfs["profundidad"] == 3
assert resultado_dfs["costo"] == 8

assert resultado_iddfs["ruta"] == ["S", "A", "G"]
assert resultado_iddfs["profundidad"] == 2
assert resultado_iddfs["costo"] == 11

# Inicio igual a meta.
assert dfs(GRAFO_EJEMPLO, "S", "S")["ruta"] == ["S"]
assert iddfs(GRAFO_EJEMPLO, "S", "S")["ruta"] == ["S"]

# Los límites 0 y 1 todavía no alcanzan G.
assert busqueda_limitada(GRAFO_EJEMPLO, "S", "G", 0)[0] is None
assert busqueda_limitada(GRAFO_EJEMPLO, "S", "G", 1)[0] is None
assert busqueda_limitada(GRAFO_EJEMPLO, "S", "G", 2)[0]["ruta"] == ["S", "A", "G"]

# Meta inalcanzable. El ciclo D → S no debe causar un recorrido infinito.
assert dfs(GRAFO_EJEMPLO, "S", "X") is None
assert iddfs(GRAFO_EJEMPLO, "S", "X") is None

print("Todas las pruebas del grafo común pasaron.")
mostrar("DFS en el grafo común", resultado_dfs)
mostrar("IDDFS en el grafo común", resultado_iddfs)

Todas las pruebas del grafo común pasaron.

DFS en el grafo común
Ruta: S → B → D → G
Profundidad: 3
Costo: 8
Nodos expandidos: 3
Tamaño máximo de la frontera: 2

IDDFS en el grafo común
Ruta: S → A → G
Profundidad: 2
Costo: 11
Nodos expandidos: 3
Tamaño máximo de la frontera: 3
Límite en que se encontró la meta: 2
